# BiPAL and baselines on the leakage-free folds

This notebook trains the proposed **BiPAL** model (its whole ablation ladder) and a set of
**baselines** on the leakage-free 5-fold splits from the audit. Families of copies and same-patient
slices never cross folds. Everything except the model is the same as the T-PAL re-run:

| Item | Setting (T-PAL's own, from the original notebook) |
|---|---|
| Images | Grayscale, bilinear resize to 128 × 128, no cropping |
| Normalisation | Mean and SD of each run's own training images |
| Augmentation | Horizontal flip; affine (±10°, ±5% shift, 0.95–1.05 scale) |
| Training | 30 epochs, batch 64, AdamW (weight decay 0.05), one-cycle schedule (15% warm-up), label smoothing 0.05, gradient clipping 1.0 |
| Checkpoint | Best validation accuracy |
| Splits | `grouped_5fold`: test = fold k, validation = fold k+1, training = the other three |
| Seeds | 42, 43, 44 (the folds stay fixed; only the training seed changes) |

So every model can be compared with T-PAL **image by image** on the same 7,200 test predictions
per seed.

**Models**

| Name | Model | Pre-trained | Peak learning rate |
|---|---|---|---|
| `bipal_tpal_like` … `bipal_mirror` | BiPAL ablation rungs (`bipal.ABLATIONS`), depth 4 | no | 1e-3 (T-PAL's) |
| `bipal_full` | Full BiPAL: mixture prior, mirror pathway, class-attention pooling | no | 1e-3 |
| `bipal_lesion` | Full BiPAL + lesion-evidence loss (λ = 0.5) on images with a figshare mask | no | 1e-3 |
| `bipal_tpal_cls`, `bipal_tpal_cls_lesion` | The T-PAL-like rung with class-attention pooling, without and with the lesion loss: does the lesion gain need BiPAL's prior? | no | 1e-3 |
| `deit_small_cls`, `deit_small_cls_lesion` | DeiT-S with BiPAL's class-attention pooling over its patch tokens (instead of the CLS head), without and with the lesion loss: does the lesion loss add to pre-training? | ImageNet-1k | 1e-4 |
| `efficientnet_b0_cls`, `efficientnet_b0_cls_lesion` | EfficientNet-B0 with output stride 16 (the last stride-2 stage dilated, same weights), so its feature map is 8 × 8 like DeiT-S's patch grid, and class-attention pooling over that map instead of global average pooling; without and with the lesion loss | ImageNet-1k | 3e-4 |
| `efficientnet_b0_aux_lesion` | The standard EfficientNet-B0 (its own head makes every prediction) plus an auxiliary class-attention branch on the stride-16 map after `blocks.4` (8 × 8), trained only by the lesion loss. Its control is the standard `efficientnet_b0`: without the loss (`efficientnet_b0_aux`) the branch gets no gradient. | ImageNet-1k | 3e-4 |
| `<lesion model>_m10`, `_m25`, `_m50` | A lesion model with masks for only 10%, 25% or 50% of the training families that have one (whole patients, nested: the 10% set is inside the 25% set, and so on). The label-efficiency curve. | as the base model | as the base model |
| `resnet18_scratch` | ResNet-18 | no | 1e-3 |
| `resnet18`, `resnet50`, `efficientnet_b0`, `mobilenetv3_large` | timm CNNs | ImageNet-1k | 3e-4 |
| `deit_small` | DeiT-S/16 (position embedding resized to 8 × 8 patches) | ImageNet-1k | 1e-4 |
| `convnext_tiny` | ConvNeXt-T (not in the default list: very slow on Apple GPUs; see `models_kaggle_gpu.ipynb`) | ImageNet-1k | 3e-4 |
| `swin_tiny` | Swin-T, 8 × 8 windows at 128 px (not in the default list; see `models_kaggle_gpu.ipynb`) | ImageNet-1k | 1e-4 |
| `<model>_lr<value>` | A learning-rate trial, e.g. `efficientnet_b0_lr1e-3`: the model with that peak learning rate instead of its default. Goes last in the name. | as the base model | the value |

Pre-trained models get the grayscale image repeated to 3 channels and keep their original input
layer. The learning rates are standard fine-tuning defaults, **not tuned**. Part E3 of the review
asks for equal tuning trials per model before the final comparison.

**Running it**
- **On Kaggle:** run the audit notebook and the T-PAL re-run first. Then **Add Input**: the dataset
  `masoudnickparvar/brain-tumor-mri-dataset`, the audit notebook's output, and (for the paired
  comparison) the T-PAL re-run's output. Turn on a GPU, and turn on Internet for the pre-trained
  weights.
- **On a Mac:** see `kaggle/README.md`. Set the folders with `MODELS_DATA_ROOT`, `MODELS_AUDIT_DIR`,
  `MODELS_TPAL_DIR` and `MODELS_OUT`.
- Set `QUICK_TEST = True` (or `MODELS_QUICK=1`) first. It trains every model for 1 epoch on 1 fold.
- Runs are ordered seed by seed (all models for seed 42, then 43, then 44). An interrupted run
  resumes where it stopped: finished runs are skipped.

In [ ]:
import os, sys, json, time, random, glob, re, gc, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
env = os.environ.get

QUICK_TEST = False   # True: 1 epoch, 1 seed, 1 fold per model, to check the notebook in a few minutes
QUICK_TEST = QUICK_TEST or env("MODELS_QUICK") == "1"

DEFAULT_MODELS = ("resnet18,resnet50,efficientnet_b0,mobilenetv3_large,deit_small,resnet18_scratch,"
                  "bipal_full,bipal_lesion,bipal_tpal_like,bipal_learn_sigma,bipal_adaptive,bipal_global,bipal_mirror,"
                  "bipal_tpal_cls_lesion,deit_small_cls_lesion,bipal_tpal_cls,deit_small_cls,"
                  "efficientnet_b0_cls_lesion,efficientnet_b0_cls,"
                  "bipal_tpal_cls_lesion_m10,bipal_tpal_cls_lesion_m25,bipal_tpal_cls_lesion_m50,"
                  "efficientnet_b0_aux_lesion,deit_small_cls_lesion_m10,deit_small_cls_lesion_m25,deit_small_cls_lesion_m50")
MODELS = env("MODELS", DEFAULT_MODELS).split(",")
PROTOCOLS = env("MODELS_PROTOCOLS", "grouped_5fold").split(",")   # add random_5fold to measure each model's leakage inflation
SEEDS = [int(s) for s in env("MODELS_SEEDS", "42,43,44").split(",")]
FOLDS = [int(k) for k in env("MODELS_FOLDS", "0,1,2,3,4").split(",")]
EPOCHS = int(env("MODELS_EPOCHS", "30"))
BS = int(env("MODELS_BS", "64"))
NUM_WORKERS = int(env("MODELS_WORKERS", "2"))
SAVE_CKPT = env("MODELS_SAVE_CKPT", "1") == "1"   # best weights per run (float16), for the XAI and calibration analyses
OUT = env("MODELS_OUT", "/kaggle/working/models_grouped")
AUDIT_DIR = env("MODELS_AUDIT_DIR")   # found under /kaggle/input when not set
DATA_ROOT = env("MODELS_DATA_ROOT")
TPAL_DIR = env("MODELS_TPAL_DIR")     # the T-PAL re-run's output folder (results.csv, preds/); optional
CKPT_DIR = env("MODELS_CKPT_DIR")     # default OUT/ckpt; about 160 MB per fold and seed for the default models
if QUICK_TEST:
    EPOCHS, SEEDS, FOLDS = 1, SEEDS[:1], FOLDS[:1]
    OUT = OUT + "_quicktest"
    CKPT_DIR = CKPT_DIR and CKPT_DIR + "_quicktest"
CKPT_DIR = CKPT_DIR or os.path.join(OUT, "ckpt")
for d in (os.path.join(OUT, "preds"), CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print("models:", MODELS)
print("protocols:", PROTOCOLS, "| seeds:", SEEDS, "| folds:", FOLDS, "| epochs:", EPOCHS, "| out:", OUT)

## 1. Project code
`code/evaluation.py` and `code/bipal.py` from the project, unchanged.

In [ ]:
%%writefile evaluation.py
"""
Evaluation utilities for a Q1-grade brain-tumour MRI classification study.

  statistics     cluster (patient/family) bootstrap CIs, paired bootstrap differences, exact McNemar
  calibration    temperature scaling, equal-width and equal-mass ECE, Brier score
  uncertainty    split-conformal prediction sets (APS / RAPS), risk-coverage curve and AURC
  explanations   localisation against tumour masks (pointing game, energy pointing game, top-k IoU, AP)
                 and patch-wise MoRF/LeRF perturbation curves with a blurred baseline

All functions take numpy arrays: y (n,), prob (n, C) softmax outputs, groups (n,) patient or
duplicate-family ids. Resample and calibrate at the group level: slices of one patient are not
exchangeable with each other.
"""
from __future__ import annotations

import numpy as np
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import average_precision_score


# ------------------------------------------------------------------------------------------ statistics

def _group_index(groups, n):
    groups = np.arange(n) if groups is None else np.asarray(groups)
    uniq, inv = np.unique(groups, return_inverse=True)
    members = [np.flatnonzero(inv == g) for g in range(len(uniq))]
    return members


def cluster_bootstrap_ci(metric, y, prob, groups=None, n_boot=2000, alpha=0.05, seed=0):
    """Point estimate and percentile CI of metric(y, prob), resampling whole groups."""
    rng = np.random.default_rng(seed)
    members = _group_index(groups, len(y))
    vals = []
    for _ in range(n_boot):
        idx = np.concatenate([members[g] for g in rng.integers(0, len(members), len(members))])
        try:
            vals.append(metric(y[idx], prob[idx]))
        except ValueError:  # e.g. a class missing from the resample
            continue
    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return metric(y, prob), lo, hi


def paired_bootstrap_diff(metric, y, prob_a, prob_b, groups=None, n_boot=2000, alpha=0.05, seed=0):
    """CI and two-sided p-value for metric(a) - metric(b) on the same test cases."""
    rng = np.random.default_rng(seed)
    members = _group_index(groups, len(y))
    diffs = []
    for _ in range(n_boot):
        idx = np.concatenate([members[g] for g in rng.integers(0, len(members), len(members))])
        try:
            diffs.append(metric(y[idx], prob_a[idx]) - metric(y[idx], prob_b[idx]))
        except ValueError:
            continue
    diffs = np.asarray(diffs)
    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    p = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    return metric(y, prob_a) - metric(y, prob_b), lo, hi, p


def mcnemar_exact(y, pred_a, pred_b):
    """Exact McNemar test on paired predictions; returns (a right & b wrong, a wrong & b right, p)."""
    a_ok, b_ok = pred_a == y, pred_b == y
    n01, n10 = int(np.sum(a_ok & ~b_ok)), int(np.sum(~a_ok & b_ok))
    p = stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return n01, n10, p


# ----------------------------------------------------------------------------------------- calibration

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def fit_temperature(logits_val, y_val):
    """Temperature T minimising validation NLL; apply as softmax(logits / T)."""
    def nll(t):
        z = logits_val / t
        z = z - z.max(1, keepdims=True)
        return -(z[np.arange(len(y_val)), y_val] - np.log(np.exp(z).sum(1))).mean()
    return minimize_scalar(nll, bounds=(0.05, 20.0), method="bounded").x


def ece(prob, y, n_bins=15, adaptive=False):
    """Top-label ECE with equal-width bins, or equal-mass bins when adaptive=True."""
    conf = prob.max(1)
    correct = (prob.argmax(1) == y).astype(float)
    if adaptive:
        edges = np.quantile(conf, np.linspace(0, 1, n_bins + 1))
    else:
        edges = np.linspace(0, 1, n_bins + 1)
    ids = np.clip(np.searchsorted(edges, conf, side="right") - 1, 0, n_bins - 1)
    out = 0.0
    for b in range(n_bins):
        m = ids == b
        if m.any():
            out += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return out


def brier(prob, y):
    onehot = np.eye(prob.shape[1])[y]
    return ((prob - onehot) ** 2).sum(1).mean()


# ----------------------------------------------------------------------------------------- uncertainty

def _aps_scores(prob, lam, k_reg, u):
    """APS/RAPS conformity score of every label (Angelopoulos et al., ICLR 2021)."""
    n, C = prob.shape
    order = np.argsort(-prob, 1)
    ps = np.take_along_axis(prob, order, 1)
    penalty = lam * np.maximum(0, np.arange(1, C + 1) - k_reg)[None]
    sorted_scores = np.cumsum(ps, 1) - u[:, None] * ps + penalty
    ranks = np.argsort(order, 1)
    return np.take_along_axis(sorted_scores, ranks, 1)


def conformal_calibrate(prob_cal, y_cal, alpha=0.1, lam=0.0, k_reg=0, seed=0):
    """Threshold qhat for (1 - alpha) marginal coverage. lam > 0 gives RAPS, lam = 0 gives APS."""
    u = np.random.default_rng(seed).uniform(size=len(y_cal))
    s = _aps_scores(prob_cal, lam, k_reg, u)[np.arange(len(y_cal)), y_cal]
    level = min(1.0, np.ceil((len(s) + 1) * (1 - alpha)) / len(s))
    return np.quantile(s, level, method="higher")


def conformal_predict(prob, qhat, lam=0.0, k_reg=0, seed=1, allow_empty=True):
    """Boolean (n, C) prediction sets. allow_empty=False always adds the top-1 label, which makes
    the sets non-empty but over-covers (coverage above 1 - alpha)."""
    u = np.random.default_rng(seed).uniform(size=len(prob))
    sets = _aps_scores(prob, lam, k_reg, u) <= qhat
    if not allow_empty:
        sets[np.arange(len(prob)), prob.argmax(1)] = True
    return sets


def set_metrics(sets, y):
    covered = sets[np.arange(len(y)), y]
    per_class = {int(c): covered[y == c].mean() for c in np.unique(y)}
    return {"coverage": covered.mean(), "avg_size": sets.sum(1).mean(),
            "singleton_rate": (sets.sum(1) == 1).mean(), "class_coverage": per_class}


def risk_coverage(prob, y):
    """Selective-prediction curve ordered by max-softmax confidence; returns coverage, risk, AURC."""
    conf = prob.max(1)
    err = (prob.argmax(1) != y).astype(float)[np.argsort(-conf)]
    k = np.arange(1, len(y) + 1)
    risk = np.cumsum(err) / k
    return k / len(y), risk, risk.mean()


# ---------------------------------------------------------------------------------------- explanations

def localisation_metrics(heat, mask, top_frac=0.05):
    """Agreement of a heat map with a binary tumour mask, both (H, W) at the same resolution.

    pointing_hit  the arg-max pixel lies inside the tumour
    ebpg          share of positive heat inside the tumour (energy-based pointing game)
    iou_top       IoU between the top top_frac pixels and the tumour
    heat_ap       average precision of heat as a per-pixel tumour detector; compare with mask_frac (chance)
    """
    heat = np.asarray(heat, dtype=float)
    m = np.asarray(mask).astype(bool)
    if not m.any():
        return None
    pos = np.clip(heat, 0, None)
    k = max(1, int(top_frac * heat.size))
    top = heat >= np.partition(heat.ravel(), -k)[-k]
    return {
        "pointing_hit": float(m.flat[np.argmax(heat)]),
        "ebpg": float(pos[m].sum() / (pos.sum() + 1e-12)),
        "iou_top": float((top & m).sum() / (top | m).sum()),
        "heat_ap": float(average_precision_score(m.ravel(), heat.ravel())),
        "mask_frac": float(m.mean()),
    }


def perturbation_curve(predict_proba, x, cls, heat, patch=8, steps=16, order="morf", rng=None):
    """Patch-wise perturbation curve with a heavily blurred baseline instead of black pixels.

    predict_proba: callable mapping a torch batch (B, C, H, W) to numpy probabilities (B, K).
    order: "morf" (most relevant first), "lerf" (least relevant first) or "random".
    Report AUC(LeRF) - AUC(MoRF) and check that MoRF beats the random order; when random removal
    hurts more than MoRF (as in the T-PAL notebook) the curve is measuring distribution shift.
    For the final paper use ROAD's noisy linear imputation (github.com/tleemann/road_evaluation).
    """
    import torch
    import torch.nn.functional as F

    H, W = x.shape[-2:]
    base = F.interpolate(F.avg_pool2d(x, 16), size=(H, W), mode="bilinear", align_corners=False)
    scores = F.avg_pool2d(torch.as_tensor(heat, dtype=x.dtype, device=x.device)[None, None], patch).flatten()
    if order == "random":
        rank = torch.randperm(len(scores), generator=rng)
    else:
        rank = torch.argsort(scores, descending=(order == "morf"))
    gw = W // patch
    fracs = np.linspace(0, 1, steps + 1)
    batch = []
    for f in fracs:
        keep = torch.ones(len(scores), device=x.device)
        keep[rank[: int(round(f * len(scores)))]] = 0
        m = keep.view(1, 1, H // patch, gw)
        m = F.interpolate(m, size=(H, W), mode="nearest")
        batch.append(x * m + base * (1 - m))
    p = predict_proba(torch.cat(batch))[:, cls]
    trapezoid = getattr(np, "trapezoid", None) or np.trapz  # numpy < 2.0 on older Kaggle images
    return fracs, p, float(trapezoid(p, fracs))

In [ ]:
%%writefile bipal.py
"""
BiPAL: Bilateral, adaptive Pixel-Association Learning (proposed successor to T-PAL).

Identity this module builds on (checked numerically in smoke_test.py):

    T-PAL:  A~ = rownorm(softmax(QK^T / sqrt(d)) * S)  ==  softmax(QK^T / sqrt(d) + log S)

so T-PAL's "pixel association" is attention with a fixed Gaussian log-distance bias. BiPAL keeps
that form but replaces the fixed isotropic S with a learnable per-head mixture prior

    P_h(j | i) = pi_loc * N(p_j ; p_i, s_i^2 Sigma_h)                    adaptive anisotropic local
               + pi_mir * gamma * N(p_j ; m(p_i), tau_h^2 I)              contralateral, learned midline
               + (pi_glob + pi_mir * (1 - gamma)) / N                     global escape

and adds a contralateral asymmetry pathway, class-specific attention pooling (an intrinsic evidence
map per class), an optional lesion-evidence loss for images that have tumour masks, a self-supervised
midline loss, and an O(k N^2) estimate of T-PAL's O(N^3) head-diversity loss.

T-PAL is recovered with  mirror=False, adaptive=False, global_escape=False, learn_prior=False,
pool="mean"  (up to T-PAL's post-LN ordering).
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def grid_coords(g: int, device=None) -> torch.Tensor:
    """(g*g, 2) token centres as (x, y) in [-1, 1], row-major like T-PAL's grid_points."""
    lin = torch.linspace(-1.0, 1.0, g, device=device)
    yy, xx = torch.meshgrid(lin, lin, indexing="ij")
    return torch.stack([xx.flatten(), yy.flatten()], -1)


def reflect(p: torch.Tensor, phi: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Reflect points p (N, 2) across the line x*cos(phi) + y*sin(phi) = c, one line per image -> (B, N, 2)."""
    n = torch.stack([torch.cos(phi), torch.sin(phi)], -1)  # (B, 2)
    dist = p[None] @ n[..., None]  # (B, N, 1)
    return p[None] - 2.0 * (dist - c[:, None, None]) * n[:, None, :]


def mirror_distances(p: torch.Tensor, mid) -> torch.Tensor:
    """(B, N, N) squared distances |p_j - m(p_i)|^2 from each token's mirror image to every token.

    Depends only on the midline, so BiPAL computes it once per forward pass and shares it with every block.
    """
    phi, c, _ = mid
    m = reflect(p, phi.float(), c.float())
    return ((p[None, None] - m[:, :, None]) ** 2).sum(-1)


def conv_bn(i, o, stride=2):
    return nn.Sequential(nn.Conv2d(i, o, 3, stride, 1, bias=False), nn.BatchNorm2d(o), nn.GELU())


class MidlineHead(nn.Module):
    """Per-image mid-sagittal line (angle phi, offset c) and symmetry confidence gamma.

    Starts at the vertical centre line with gamma = 0.5. gamma lets the model switch the
    contralateral pathway off for views without left-right symmetry (sagittal slices).
    """

    def __init__(self, d, max_angle=math.radians(20), max_offset=0.3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, d // 2), nn.GELU(), nn.Linear(d // 2, 3))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)
        self.max_angle, self.max_offset = max_angle, max_offset

    def forward(self, feat):
        raw = self.net(feat)
        phi = self.max_angle * torch.tanh(raw[:, 0])
        c = self.max_offset * torch.tanh(raw[:, 1])
        gamma = torch.sigmoid(raw[:, 2])
        return phi, c, gamma


class AssociationPrior(nn.Module):
    """log P_h(j | i) for every head, shape (B, h, N, N); rows of P sum to 1."""

    def __init__(self, d, heads, grid, sigma0=3.0, tau0=1.5, adaptive=True, mirror=True,
                 global_escape=True, learn_prior=True):
        super().__init__()
        self.h, self.mirror, self.global_escape = heads, mirror, global_escape
        self.register_buffer("pts", grid_coords(grid))
        unit = 2.0 / (grid - 1)  # one token step in normalised coordinates
        self.log_sx = nn.Parameter(torch.full((heads,), math.log(sigma0 * unit)), requires_grad=learn_prior)
        self.log_sy = nn.Parameter(torch.full((heads,), math.log(sigma0 * unit)), requires_grad=learn_prior)
        self.theta = nn.Parameter(torch.zeros(heads), requires_grad=learn_prior)
        self.log_tau = nn.Parameter(torch.full((heads,), math.log(tau0 * unit)), requires_grad=learn_prior)
        self.adapt = nn.Linear(d, heads) if adaptive else None
        if self.adapt is not None:  # per-query scale s_i in [1/e, e], starts at 1
            nn.init.zeros_(self.adapt.weight)
            nn.init.zeros_(self.adapt.bias)
        # mixture over {local, mirror, global}; starts mostly local, like T-PAL
        self.mix = nn.Parameter(torch.tensor([3.0, 0.0, 0.0]).repeat(heads, 1))
        ninf = float("-inf")
        self.register_buffer("mix_mask", torch.tensor([0.0, 0.0 if mirror else ninf, 0.0 if global_escape else ninf]))

    def mixture(self):
        return torch.softmax(self.mix + self.mix_mask, -1)  # (h, 3)

    def forward(self, x, mid=None, d2=None):
        """d2: optional precomputed mirror_distances(self.pts, mid), shared across blocks."""
        B, N, _ = x.shape
        p = self.pts
        delta = p[None, :, :] - p[:, None, :]  # (N, N, 2) = p_j - p_i
        ct, st = torch.cos(self.theta)[:, None, None], torch.sin(self.theta)[:, None, None]
        u = delta[..., 0] * ct + delta[..., 1] * st  # (h, N, N) along each head's principal axes
        v = -delta[..., 0] * st + delta[..., 1] * ct
        q = (u / self.log_sx.exp()[:, None, None]) ** 2 + (v / self.log_sy.exp()[:, None, None]) ** 2
        q = q[None].expand(B, -1, -1, -1)
        if self.adapt is not None:
            s = torch.exp(torch.tanh(self.adapt(x).float()))  # (B, N, h)
            q = q / s.permute(0, 2, 1)[..., None] ** 2
        if not (self.mirror or self.global_escape):
            return torch.log_softmax(-0.5 * q, dim=-1)  # normalised local kernel

        # Mix the components as probabilities and take one log at the end. This gives the same values as
        # chaining logaddexp in log space, at a fraction of the cost. Every row keeps a strictly positive
        # floor (the global term, or the mirror term's (1 - gamma) / N share), so the log is finite.
        pi = self.mixture()
        w = lambda k: pi[:, k][None, :, None, None]
        P = w(0) * torch.softmax(-0.5 * q, dim=-1)
        if self.mirror and mid is not None:
            gamma = mid[2].float().clamp(1e-4, 1 - 1e-4)[:, None, None, None]
            if d2 is None:
                d2 = mirror_distances(p, mid)
            P_mir = torch.softmax(-0.5 * d2[:, None] / self.log_tau.exp()[None, :, None, None] ** 2, -1)
            P = P + w(1) * (gamma * P_mir + (1.0 - gamma) / N)
        if self.global_escape:
            P = P + w(2) / N
        return torch.log(P.clamp_min(torch.finfo(P.dtype).tiny))


class BiPALBlock(nn.Module):
    def __init__(self, d, heads, grid, drop=0.1, drop_path=0.0, **prior_kw):
        super().__init__()
        self.h, self.dh = heads, d // heads
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.o = nn.Linear(d, d)
        self.prior = AssociationPrior(d, heads, grid, **prior_kw)
        self.use_mirror = self.prior.mirror
        if self.use_mirror:
            self.mq, self.mk, self.asym = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.g = nn.Linear((3 if self.use_mirror else 2) * d, d)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(4 * d, d))
        self.dp = nn.Dropout(drop)
        self.drop_path = drop_path

    def _path(self, t):
        if not self.training or self.drop_path == 0.0:
            return t
        keep = torch.rand(t.shape[0], 1, 1, device=t.device) >= self.drop_path
        return t * keep / (1.0 - self.drop_path)

    def forward(self, e, mid=None, d2=None):
        B, N, d = e.shape
        x = self.n1(e)
        q, k, v = self.qkv(x).view(B, N, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        logits = (q @ k.transpose(-1, -2)).float() / math.sqrt(self.dh)
        A = torch.softmax(logits, -1)  # content-only association, kept for analysis and L_div
        if self.use_mirror and mid is not None and d2 is None:
            d2 = mirror_distances(self.prior.pts, mid)
        At = torch.softmax(logits + self.prior(x, mid, d2), -1)
        hh = self.o((self.dp(At).to(v.dtype) @ v).transpose(1, 2).reshape(B, N, d))

        feats, asym, upd = [x, hh], None, None
        if self.use_mirror and mid is not None:
            gamma = mid[2]
            tau = self.prior.log_tau.exp().mean()
            ml = (self.mq(x) @ self.mk(x).transpose(-1, -2)).float() / math.sqrt(d) - 0.5 * d2 / tau ** 2
            contra = torch.softmax(ml, -1).to(x.dtype) @ x  # content-aware look-up around the mirrored position
            asym = gamma[:, None, None].to(x.dtype) * (x - contra)  # bilateral asymmetry features
            a = self.asym(asym)
            feats.append(a)
        g = torch.sigmoid(self.g(torch.cat(feats, -1)))
        upd = g * hh if asym is None else g * hh + (1 - g) * a
        e = e + self._path(self.dp(upd))
        e = e + self._path(self.dp(self.ff(self.n2(e))))
        return e, {"A": A, "At": At, "g": g, "asym": asym}


class ClassAttentionPool(nn.Module):
    """One learned query per class -> evidence map alpha_c over tokens, and a logit per class."""

    def __init__(self, d, nc):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.q = nn.Parameter(torch.randn(nc, d) * 0.02)
        self.w = nn.Parameter(torch.randn(nc, d) * 0.02)
        self.b = nn.Parameter(torch.zeros(nc))

    def forward(self, e):
        z = self.norm(e)
        alpha = torch.softmax(torch.einsum("bnd,cd->bcn", z, self.q).float() / math.sqrt(z.shape[-1]), -1)
        pooled = torch.einsum("bcn,bnd->bcd", alpha.to(z.dtype), z)
        return (pooled * self.w).sum(-1) + self.b, alpha, pooled


class BiPAL(nn.Module):
    def __init__(self, nc=4, img=128, in_ch=1, d=128, heads=4, depth=4, drop=0.1, drop_path=0.1,
                 mirror=True, adaptive=True, global_escape=True, learn_prior=True, sigma0=3.0, pool="class"):
        super().__init__()
        self.grid = img // 8
        self.mirror = mirror
        self.stem = nn.Sequential(conv_bn(in_ch, 32), conv_bn(32, 64), conv_bn(64, d))
        self.pos = nn.Parameter(torch.zeros(1, self.grid ** 2, d))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.midline = MidlineHead(d) if mirror else None
        prior_kw = dict(sigma0=sigma0, adaptive=adaptive, mirror=mirror, global_escape=global_escape,
                        learn_prior=learn_prior)
        dpr = torch.linspace(0, drop_path, depth).tolist()
        self.blocks = nn.ModuleList([BiPALBlock(d, heads, self.grid, drop, dpr[i], **prior_kw) for i in range(depth)])
        self.pool_type = pool
        if pool == "class":
            self.pool = ClassAttentionPool(d, nc)
        else:
            self.norm = nn.LayerNorm(d)
            self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(drop), nn.Linear(d, nc))

    def forward(self, x):
        e = self.stem(x).flatten(2).transpose(1, 2) + self.pos
        mid = self.midline(e.mean(1)) if self.mirror else None
        d2 = mirror_distances(self.blocks[0].prior.pts, mid) if self.mirror else None  # shared by all blocks
        aux = {"A": [], "At": [], "g": [], "asym": [], "mid": mid}
        for blk in self.blocks:
            e, a = blk(e, mid, d2)
            for k in ("A", "At", "g", "asym"):
                aux[k].append(a[k])
        if self.pool_type == "class":
            logits, alpha, pooled = self.pool(e)
            aux.update(alpha=alpha, feat=pooled.mean(1))
        else:
            feat = self.norm(e).mean(1)
            logits = self.head(feat)
            aux.update(alpha=None, feat=feat)
        aux["tokens"] = e
        return logits, aux


# ----------------------------------------------------------------------------------------------- losses

def lesion_evidence_loss(alpha, y, masks, has_mask, grid):
    """-log(share of the true-class evidence map that lies on tumour tokens), for images with a mask.

    alpha: (B, C, N) class evidence maps; masks: (B, 1, H, W) binary; has_mask: (B,) float in {0, 1}.
    Images without a mask (and no-tumour images) contribute nothing.
    """
    if has_mask.sum() == 0:
        return alpha.new_zeros(())
    m = F.adaptive_max_pool2d(masks.float(), grid).flatten(1)  # token touches tumour
    a = alpha[torch.arange(len(y), device=y.device), y]
    frac = (a * m).sum(-1).clamp_min(1e-6)
    return -(torch.log(frac) * has_mask).sum() / has_mask.sum()


def head_diversity_loss(attn_list, probes=8):
    """Hutchinson estimate of T-PAL's mean_{a<b} ||A_a A_b^T||_F^2 in O(k N^2) instead of O(N^3).

    Uses ||M||_F^2 = E_z ||M z||^2 for z ~ N(0, I), with M = A_a A_b^T, so M z = A_a (A_b^T z).
    """
    total = 0.0
    for A in attn_list:
        A = A.float()
        B, h, N, _ = A.shape
        z = torch.randn(B, 1, N, probes, device=A.device)
        W = A.transpose(-1, -2) @ z  # (B, h, N, k)
        MZ = torch.einsum("bain,bcnk->backi", A, W)  # (B, h, h, k, N)
        sq = MZ.pow(2).sum(-1).mean(-1)  # (B, h, h)
        iu = torch.triu_indices(h, h, 1, device=A.device)
        total = total + sq[:, iu[0], iu[1]].mean()
    return total / len(attn_list)


def midline_symmetry_loss(x, mid, size=64):
    """Self-supervised midline fit: reflecting the image about the predicted line should reproduce it.

    Weighted by the (detached) symmetry confidence so sagittal views are not forced to be symmetric.
    """
    phi, c, gamma = mid
    xs = F.interpolate(x.float(), size=(size, size), mode="bilinear", align_corners=True)
    grid = reflect(grid_coords(size, x.device), phi.float(), c.float()).view(-1, size, size, 2)
    xr = F.grid_sample(xs, grid, mode="bilinear", padding_mode="border", align_corners=True)
    fg = (xs > xs.amin((2, 3), keepdim=True) + 1e-3).float()
    res = ((xs - xr).abs() * fg).sum((1, 2, 3)) / fg.sum((1, 2, 3)).clamp_min(1.0)
    return (gamma.detach().float() * res).mean()


def bipal_loss(model, logits, aux, x, y, masks=None, has_mask=None, ce=None,
               lam_exp=0.5, lam_div=0.01, lam_mid=0.1):
    ce = ce or nn.CrossEntropyLoss(label_smoothing=0.05)
    parts = {"cls": ce(logits.float(), y)}
    if lam_exp and masks is not None and aux.get("alpha") is not None:
        parts["exp"] = lesion_evidence_loss(aux["alpha"], y, masks, has_mask, model.grid)
    if lam_div:
        parts["div"] = head_diversity_loss(aux["A"])
    if lam_mid and aux.get("mid") is not None:
        parts["mid"] = midline_symmetry_loss(x, aux["mid"])
    w = {"cls": 1.0, "exp": lam_exp, "div": lam_div, "mid": lam_mid}
    total = sum(w[k] * v for k, v in parts.items())
    return total, {k: float(v.detach()) for k, v in parts.items()}


# ------------------------------------------------------------------------------------- ablation ladder

ABLATIONS = {
    # name: kwargs for BiPAL; run each with the same data, splits, seeds and budget
    "tpal_like": dict(mirror=False, adaptive=False, global_escape=False, learn_prior=False, pool="mean"),
    "+learn_sigma": dict(mirror=False, adaptive=False, global_escape=False, learn_prior=True, pool="mean"),
    "+adaptive": dict(mirror=False, adaptive=True, global_escape=False, learn_prior=True, pool="mean"),
    "+global": dict(mirror=False, adaptive=True, global_escape=True, learn_prior=True, pool="mean"),
    "+mirror": dict(mirror=True, adaptive=True, global_escape=True, learn_prior=True, pool="mean"),
    "+class_pool": dict(mirror=True, adaptive=True, global_escape=True, learn_prior=True, pool="class"),
    # "+lesion_loss" = "+class_pool" trained with lam_exp > 0 on the masked subset (vary the mask fraction)
}

## 2. Find the data and the audit output, and build the splits

In [ ]:
def find_dir(base, test):
    for d, subdirs, files in os.walk(base):
        if test(d, subdirs, files):
            return d
    return None

DATA_ROOT = DATA_ROOT or find_dir("/kaggle/input", lambda d, s, f: {"Training", "Testing"} <= set(s))
AUDIT_DIR = AUDIT_DIR or find_dir("/kaggle/input", lambda d, s, f: "grouped_folds.csv" in f)
TPAL_DIR = TPAL_DIR or find_dir("/kaggle/input", lambda d, s, f: "results.csv" in f and "preds" in s)
assert DATA_ROOT, "Add the dataset masoudnickparvar/brain-tumor-mri-dataset (Add Input)."
assert AUDIT_DIR, "Add the leakage-audit notebook's output (Add Input -> Your Work)."
print("dataset:", DATA_ROOT)
print("audit:  ", AUDIT_DIR)
print("T-PAL:  ", TPAL_DIR or "not found: the paired comparison with T-PAL will be skipped")

df = pd.read_csv(os.path.join(AUDIT_DIR, "grouped_folds.csv"))
df["image_path"] = [os.path.join(DATA_ROOT, p) for p in df.rel_path]
# Masks: use the audit folder's copy, so absolute paths saved on another machine do not matter.
df["mask_file"] = [os.path.join(AUDIT_DIR, "masks", os.path.basename(p)) if isinstance(p, str) else None
                   for p in df.mask_path]
missing = int((~df.image_path.map(os.path.exists)).sum())
assert missing == 0, f"{missing} images in the audit are missing here: is this the same dataset version?"
classes = sorted(df.label.unique())
NC = len(classes)
cmap_ = {c: i for i, c in enumerate(classes)}
y_all = df.label.map(cmap_).values
fam_all = df.family.values
source_all = np.where(df.patient.notna(), "figshare", "other")
print(f"{len(df):,} images, {df.family.nunique():,} families, {df.mask_file.notna().sum():,} with a tumour mask")

In [ ]:
from sklearn.model_selection import StratifiedKFold

SPLIT_SEED = 42  # identical to the T-PAL re-run, so the folds match image for image
df["fold_random"] = -1
for k, (_, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SPLIT_SEED).split(df, y_all)):
    df.loc[df.index[te], "fold_random"] = k

# Held-out source (RESULTS.md §8): the leakage-free folds, but every tumour image that does not come from figshare
# is taken out of training and validation and added to every test set. The models then learn tumours only from
# figshare, while all no-tumour images come from the other sources: the source shortcut is as strong as it can be,
# and the other-source tumours test whether a model recognises a tumour it has never seen from that source.
OTHER_TUMOUR = (source_all == "other") & (df.label.values != "notumor")

def split(protocol, fold):
    f = df[{"random_5fold": "fold_random", "grouped_5fold": "fold", "heldout_source": "fold"}[protocol]].values
    va_fold = (fold + 1) % 5
    if protocol == "heldout_source":
        return (np.where(~np.isin(f, [fold, va_fold]) & ~OTHER_TUMOUR)[0], np.where((f == va_fold) & ~OTHER_TUMOUR)[0],
                np.where((f == fold) | OTHER_TUMOUR)[0])
    return np.where(~np.isin(f, [fold, va_fold]))[0], np.where(f == va_fold)[0], np.where(f == fold)[0]

RUNS = [(m, p, k, s) for s in SEEDS for m in MODELS for p in PROTOCOLS for k in FOLDS]
for p in PROTOCOLS:
    tr, va, te = split(p, 0)
    print(f"{p}, fold 0: train {len(tr)}, val {len(va)}, test {len(te)}; "
          f"test images with a family member in train: {np.isin(fam_all[te], fam_all[tr]).mean():.1%}")
print(f"{len(RUNS)} runs planned")

In [ ]:
from PIL import Image
from tqdm.auto import tqdm

IMG = 128
def load_img(p):  # T-PAL's loader: grayscale, bilinear resize to 128 x 128, no cropping
    return np.array(Image.open(p).convert("L").resize((IMG, IMG), Image.BILINEAR), dtype=np.uint8)

def load_mask(p):  # box filter, then any coverage counts as tumour (keeps small tumours)
    if not isinstance(p, str):
        return np.zeros((IMG, IMG), np.uint8)
    m = np.array(Image.open(p).convert("L").resize((IMG, IMG), Image.BOX))
    return (m > 0).astype(np.uint8)

X_all = np.stack([load_img(p) for p in tqdm(df.image_path, desc="Images")])
M_all = np.stack([load_mask(p) for p in tqdm(df.mask_file, desc="Masks")])
H_all = df.mask_file.notna().values.astype(np.float32)
print(X_all.shape, M_all.shape, f"mean tumour fraction on masked images: {M_all[H_all > 0].mean():.3f}")

## 3. Models and the training loop
The training loop is T-PAL's (same optimiser, schedule, clipping, label smoothing and best-validation
checkpoint). BiPAL is trained with `bipal.bipal_loss`: cross-entropy, the head-diversity term
(λ = 0.01, as T-PAL's L_div), the midline term (λ = 0.1) for rungs with the mirror pathway, and, for
the `*_lesion` models only, the lesion-evidence term (λ = 0.5). The DeiT-S and EfficientNet-B0 models
with class-attention pooling get cross-entropy plus, for the `*_cls_lesion` ones, the same lesion term
on their 8 × 8 grid. A `_mNN` suffix keeps the masks of only NN% of the training families that have
one; the other images train without the lesion term. The augmentation transforms each image and its
mask together.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss, roc_auc_score
from sklearn.preprocessing import label_binarize

sys.path.insert(0, os.getcwd())
import evaluation as ev
import bipal

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print("device:", device)

LR_SCRATCH, LR_CNN, LR_VIT, WD, SMOOTH = 1e-3, 3e-4, 1e-4, 0.05, 0.05
LAM_DIV, LAM_MID, LAM_EXP = 0.01, 0.1, 0.5
RUNGS = {n: bipal.ABLATIONS[r] for n, r in {
    "bipal_tpal_like": "tpal_like", "bipal_learn_sigma": "+learn_sigma", "bipal_adaptive": "+adaptive",
    "bipal_global": "+global", "bipal_mirror": "+mirror", "bipal_full": "+class_pool", "bipal_lesion": "+class_pool"}.items()}
RUNGS["bipal_tpal_cls"] = RUNGS["bipal_tpal_cls_lesion"] = dict(bipal.ABLATIONS["tpal_like"], pool="class")
VIT_CLS = {"deit_small_cls": "deit_small", "deit_small_cls_lesion": "deit_small"}  # timm ViT + class-attention pooling
CNN_CLS = {"efficientnet_b0_cls": "efficientnet_b0", "efficientnet_b0_cls_lesion": "efficientnet_b0"}  # timm CNN + class-attention pooling
# Standard timm CNN (its own head makes every prediction) + an auxiliary class-attention branch on the stride-16
# map, used only by the lesion loss. Without the loss (efficientnet_b0_aux) the branch gets no gradient, so that
# model is the standard one: a sanity check, not a run to repeat.
CNN_AUX = {"efficientnet_b0_aux": ("efficientnet_b0", "blocks.4"), "efficientnet_b0_aux_lesion": ("efficientnet_b0", "blocks.4")}
# Prior-art baseline (RESULTS.md §9): GAIN with external supervision (Li et al., CVPR 2018, Eq. 8) on the standard
# EfficientNet-B0. Grad-CAM attention for the true class at the same stride-16 tap as the aux branch (8 x 8 at 128 px;
# the paper uses VGG's last conv layer, 14 x 14 at 224 px, also stride 16).
CNN_GAIN = {"efficientnet_b0_gain": ("efficientnet_b0", "blocks.4")}
GAIN_ALPHA, GAIN_OMEGA = 1.0, 10.0   # the paper's weights for the attention-mining and external-supervision terms
GAIN_SIGMA, GAIN_SCALE = 0.5, 10.0   # the soft mask T(A) = sigmoid(scale (A - sigma)); not stated in the paper: our choice
LESION = {"bipal_lesion", "bipal_tpal_cls_lesion", "deit_small_cls_lesion", "efficientnet_b0_cls_lesion",
          "efficientnet_b0_aux_lesion", "efficientnet_b0_gain"}  # trained with a lesion loss

def split_name(name):
    # "bipal_tpal_cls_lesion_m25" -> ("bipal_tpal_cls_lesion", 0.25): a lesion model with 25% of the masks
    mt = re.fullmatch(r"(.+)_m(\d+)", name)
    if mt and mt.group(1) in LESION:
        return mt.group(1), int(mt.group(2)) / 100
    return name, 1.0

def split_lr(name):
    # "efficientnet_b0_lr1e-3" -> ("efficientnet_b0", 1e-3): a learning-rate trial; it goes last in the name
    mt = re.fullmatch(r"(.+)_lr(\d+(?:\.\d+)?e-\d+)", name)
    return (mt.group(1), float(mt.group(2))) if mt else (name, None)
TIMM = {"resnet18": ("resnet18.a1_in1k", True, LR_CNN), "resnet50": ("resnet50.a1_in1k", True, LR_CNN),
        "efficientnet_b0": ("efficientnet_b0.ra_in1k", True, LR_CNN),
        "mobilenetv3_large": ("mobilenetv3_large_100.ra_in1k", True, LR_CNN),
        "convnext_tiny": ("convnext_tiny.fb_in1k", True, LR_CNN),
        "deit_small": ("deit_small_patch16_224.fb_in1k", True, LR_VIT),
        "swin_tiny": ("swin_tiny_patch4_window7_224.ms_in1k", True, LR_VIT),
        "resnet18_scratch": ("resnet18", False, LR_SCRATCH)}
unknown = [m for m in MODELS if not any(split_name(split_lr(m)[0])[0] in s for s in (RUNGS, TIMM, VIT_CLS, CNN_CLS, CNN_AUX, CNN_GAIN))]
assert not unknown, f"unknown model names: {unknown}"

class RGB(nn.Module):
    # Pre-trained timm model on a grayscale input: repeat the channel and keep the original stem.
    def __init__(self, net):
        super().__init__()
        self.net = net
    def forward(self, x):
        return self.net(x.expand(-1, 3, -1, -1)), None

class TokenClassPool(nn.Module):
    # Pre-trained timm ViT with BiPAL's class-attention pooling over its patch tokens instead of the CLS
    # head, so the lesion loss can act on the class evidence maps (grid = the patch grid, 8 x 8 here).
    def __init__(self, net):
        super().__init__()
        self.net, self.pool = net, bipal.ClassAttentionPool(net.embed_dim, NC)
        self.grid = net.patch_embed.grid_size[0]
    def forward(self, x):
        e = self.net.forward_features(x.expand(-1, 3, -1, -1))[:, self.net.num_prefix_tokens:]
        logits, alpha, _ = self.pool(e)
        return logits, {"alpha": alpha}

class MapClassPool(nn.Module):
    # Pre-trained timm CNN with BiPAL's class-attention pooling over its last feature map instead of global
    # average pooling. Built with output stride 16, so the map is 8 x 8 at 128 px, the same grid as DeiT-S.
    def __init__(self, net):
        super().__init__()
        self.net, self.pool = net, bipal.ClassAttentionPool(net.num_features, NC)
        net.eval()  # eval mode, so this probe does not touch the BatchNorm running statistics
        with torch.no_grad():
            self.grid = net.forward_features(torch.zeros(1, 3, IMG, IMG)).shape[-1]
        net.train()
    def forward(self, x):
        f = self.net.forward_features(x.expand(-1, 3, -1, -1))   # (B, C, g, g)
        logits, alpha, _ = self.pool(f.flatten(2).transpose(1, 2))
        return logits, {"alpha": alpha}

class AuxClassPool(nn.Module):
    # Pre-trained timm CNN kept exactly as it is (output stride 32, global average pooling, linear classifier:
    # its logits are the predictions), plus an auxiliary class-attention branch on an intermediate feature map
    # (`tap`; blocks.4 of EfficientNet-B0 is 112 x 8 x 8 at 128 px). Only the lesion loss reads the branch, so it
    # shapes the backbone up to the tap without changing the classifier. The branch is created under a forked
    # RNG, so the backbone, the head and the training randomness are those of the standard model.
    def __init__(self, net, tap):
        super().__init__()
        self.net, self._tap = net, None
        mod = net.get_submodule(tap)
        mod.register_forward_hook(lambda m, i, o: setattr(self, "_tap", o))
        net.eval()
        with torch.no_grad():
            net(torch.zeros(1, 3, IMG, IMG))
        net.train()
        ch, self.grid = self._tap.shape[1], self._tap.shape[-1]
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(2026)
            self.pool = bipal.ClassAttentionPool(ch, NC)
        self._tap = None
    def forward(self, x):
        logits = self.net(x.expand(-1, 3, -1, -1))
        f, self._tap = self._tap, None
        if not self.training:
            return logits, None
        _, alpha, _ = self.pool(f.flatten(2).transpose(1, 2))
        return logits, {"alpha": alpha}

class GainNet(nn.Module):
    # Pre-trained timm CNN kept exactly as it is (its logits are the predictions). In training it also returns the
    # tapped feature map, from which loss_fn builds GAIN's Grad-CAM attention (Eqs. 1-2) for the true class.
    def __init__(self, net, tap):
        super().__init__()
        self.net, self._tap = net, None
        net.get_submodule(tap).register_forward_hook(lambda m, i, o: setattr(self, "_tap", o))
        net.eval()
        with torch.no_grad():
            net(torch.zeros(1, 3, IMG, IMG))
        net.train()
        self.grid = self._tap.shape[-1]
        self._tap = None
    def forward(self, x):
        logits = self.net(x.expand(-1, 3, -1, -1))
        f, self._tap = self._tap, None
        return logits, ({"feat": f} if self.training else None)

def gain_loss(model, x, y, m, h, logits, feat):
    # GAIN_ext (Li et al. 2018): L = L_cl + alpha L_am + omega L_e, computed for the true class.
    # Attention: A = ReLU(sum_k w_k f_k), w_k = GAP(d s_y / d f_k), kept in the graph (create_graph) as in the paper.
    s_y = logits.gather(1, y[:, None]).sum()
    g = torch.autograd.grad(s_y, feat, create_graph=True)[0]
    A = F.relu((g.mean((2, 3), keepdim=True) * feat).sum(1))                     # (B, g, g)
    A = A / (A.flatten(1).amax(1).view(-1, 1, 1) + 1e-6)                        # scaled to [0, 1] per image
    # attention mining (Eqs. 3-5): erase the attended region; the true-class score on what is left should be low
    T_ = torch.sigmoid(GAIN_SCALE * (F.interpolate(A[:, None], size=x.shape[-2:], mode="bilinear", align_corners=False) - GAIN_SIGMA))
    x_star = x - T_ * x
    l_am = torch.softmax(model.net(x_star.expand(-1, 3, -1, -1)).float(), 1).gather(1, y[:, None]).mean()
    model._tap = None
    # external supervision (Eq. 7) on the images with a mask: (A - H)^2, H the mask on the same grid
    if h.sum() > 0:
        H = F.adaptive_max_pool2d(m.float(), model.grid)[:, 0]
        l_e = (((A - H) ** 2).flatten(1).mean(1) * h).sum() / h.sum()
    else:
        l_e = A.new_zeros(())
    return GAIN_ALPHA * l_am + GAIN_OMEGA * l_e

def build(name):
    if name in RUNGS:
        return bipal.BiPAL(nc=NC, img=IMG, depth=4, **RUNGS[name]), LR_SCRATCH
    import timm
    if name in VIT_CLS:
        arch, pre, lr = TIMM[VIT_CLS[name]]
        return TokenClassPool(timm.create_model(arch, pretrained=pre, num_classes=0, img_size=IMG)), lr
    if name in CNN_CLS:
        arch, pre, lr = TIMM[CNN_CLS[name]]
        return MapClassPool(timm.create_model(arch, pretrained=pre, num_classes=0, global_pool="", output_stride=16)), lr
    if name in CNN_GAIN:
        base, tap = CNN_GAIN[name]
        arch, pre, lr = TIMM[base]
        return GainNet(timm.create_model(arch, pretrained=pre, num_classes=NC), tap), lr
    if name in CNN_AUX:
        base, tap = CNN_AUX[name]
        arch, pre, lr = TIMM[base]
        return AuxClassPool(timm.create_model(arch, pretrained=pre, num_classes=NC), tap), lr
    arch, pre, lr = TIMM[name]
    # DeiT-S: position embedding resized to the 8 x 8 grid. Swin-T: 8 x 8 windows tile the 32/16/8 grids exactly
    # at 128 px (the last 4 x 4 stage uses one window); timm resizes the pre-trained relative-position tables.
    kw = dict(img_size=IMG) if arch.startswith("deit") else dict(img_size=IMG, window_size=8) if arch.startswith("swin") else {}
    return RGB(timm.create_model(arch, pretrained=pre, num_classes=NC, **kw)), lr

aug_tf = T.Compose([T.RandomHorizontalFlip(), T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05))])

class DS(Dataset):
    def __init__(self, idx, aug, mean, std, h=None):  # h: which images may use their mask (default: all that have one)
        self.X, self.M = torch.from_numpy(X_all[idx]), torch.from_numpy(M_all[idx])
        self.y = torch.from_numpy(y_all[idx]).long()
        self.h = torch.from_numpy(H_all[idx] if h is None else np.asarray(h, np.float32))
        self.aug, self.mean, self.std = aug, mean, std
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        x = self.X[i].float().div(255.0).unsqueeze(0)
        m = self.M[i].float().unsqueeze(0)
        if self.aug:  # one random flip and affine for the image and its mask
            x, m = aug_tf(torch.cat([x, m])).split(1)
        return (x - self.mean) / self.std, self.y[i], (m > 0.5).float(), self.h[i]

def loss_fn(name, model, x, y, m, h, crit):
    o, aux = model(x)
    if name in CNN_GAIN:
        return crit(o.float(), y) + gain_loss(model, x, y, m, h, o, aux["feat"])
    lam_exp = LAM_EXP if name in LESION else 0.0
    if name in VIT_CLS or name in CNN_CLS or name in CNN_AUX:
        loss = crit(o.float(), y)
        return loss + lam_exp * bipal.lesion_evidence_loss(aux["alpha"], y, m, h, model.grid) if lam_exp else loss
    if name not in RUNGS:
        return crit(o.float(), y)
    loss, _ = bipal.bipal_loss(model, o, aux, x, y, m, h, ce=crit, lam_exp=lam_exp,
                               lam_div=LAM_DIV, lam_mid=LAM_MID if model.mirror else 0.0)
    return loss

@torch.no_grad()
def predict(model, loader):
    model.eval()
    L, Y = [], []
    for x, y, _, _ in loader:
        o, _ = model(x.to(device))
        L.append(o.float().cpu())
        Y.append(y)
    return torch.cat(L), torch.cat(Y).numpy()

def ece_score(probs, y, bins=10):  # T-PAL's ECE
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (conf > lo) & (conf <= hi)
        if sel.sum() > 0:
            e += sel.mean() * abs((pred[sel] == y[sel]).mean() - conf[sel].mean())
    return e

NOTUMOR = cmap_.get("notumor")

def mask_subset(tr, frac, fold, seed):
    # Keep the masks of a fraction of the training families that have one (whole patients). The order comes
    # from one permutation per fold and seed, so the subsets are nested: 10% inside 25% inside 50% inside 100%.
    h = H_all[tr].copy()
    if frac >= 1:
        return h
    fams = np.unique(fam_all[tr][h > 0])
    order = np.random.default_rng([seed, fold, 2026]).permutation(fams)
    keep = order[:int(np.ceil(frac * len(fams)))]
    return h * np.isin(fam_all[tr], keep)

def run_one(name, protocol, fold, seed, run_id):
    tr, va, te = split(protocol, fold)
    name, lr_trial = split_lr(name)  # a learning-rate trial overrides the default peak learning rate
    name, frac = split_name(name)  # the base model, and the share of masks it may use
    h_tr = mask_subset(tr, frac, fold, seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    mean, std = float(X_all[tr].mean() / 255.0), float(X_all[tr].std() / 255.0)
    train_loader = DataLoader(DS(tr, True, mean, std, h_tr), batch_size=BS, shuffle=True, drop_last=True,
                              num_workers=NUM_WORKERS, generator=torch.Generator().manual_seed(seed))
    val_loader = DataLoader(DS(va, False, mean, std), batch_size=128, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(DS(te, False, mean, std), batch_size=128, shuffle=False, num_workers=NUM_WORKERS)

    model, lr = build(name)
    lr = lr_trial or lr
    model = model.to(device)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.15)
    crit = nn.CrossEntropyLoss(label_smoothing=SMOOTH)
    best_acc, best_state, best_ep = -1.0, None, 0
    for ep in range(EPOCHS):
        model.train()
        for x, y, m, h in train_loader:
            x, y, m, h = x.to(device), y.to(device), m.to(device), h.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(name, model, x, y, m, h, crit)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
        vl, vy = predict(model, val_loader)
        vacc = accuracy_score(vy, vl.argmax(1).numpy())
        if vacc > best_acc:
            best_acc, best_ep = vacc, ep + 1
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    if SAVE_CKPT:
        torch.save({k: (v.half() if v.is_floating_point() else v).cpu() for k, v in best_state.items()},
                   os.path.join(CKPT_DIR, run_id + ".pt"))

    lg, yt = predict(model, test_loader)
    pr = torch.softmax(lg, 1).numpy()
    pred = pr.argmax(1)
    src = source_all[te]
    m_ = {"accuracy": accuracy_score(yt, pred), "balanced_accuracy": balanced_accuracy_score(yt, pred),
          "macro_f1": f1_score(yt, pred, average="macro"), "log_loss": log_loss(yt, pr, labels=list(range(NC))),
          "ece_tpal": ece_score(pr, yt), "ece_adaptive": ev.ece(pr, yt, adaptive=True), "brier": ev.brier(pr, yt),
          "best_val_acc": best_acc, "best_epoch": best_ep, "lr": lr,
          "params_m": sum(p.numel() for p in model.parameters()) / 1e6,
          "n_train": len(tr), "n_val": len(va), "n_test": len(te),
          "mask_frac": frac, "n_train_masked": int(h_tr.sum())}
    try:
        m_["auroc_macro"] = roc_auc_score(label_binarize(yt, classes=list(range(NC))), pr, average="macro")
    except ValueError:
        m_["auroc_macro"] = np.nan
    if NOTUMOR is not None:
        tum = yt != NOTUMOR
        m_["tumour_sensitivity"] = float((pred[tum] != NOTUMOR).mean())
        m_["notumor_specificity"] = float((pred[~tum] == NOTUMOR).mean())
        for s in ("figshare", "other"):  # missed tumours by image source (the source shortcut)
            sel = tum & (src == s)
            m_[f"missed_tumours_{s}"] = float((pred[sel] == NOTUMOR).mean()) if sel.any() else np.nan
    for c, i in cmap_.items():
        m_[f"recall_{c}"] = float((pred[yt == i] == i).mean()) if (yt == i).any() else np.nan
    preds = pd.DataFrame({"rel_path": df.rel_path.values[te], "family": fam_all[te], "y": yt, "source": src})
    for i, c in enumerate(classes):
        preds[f"p_{c}"] = pr[:, i]
        preds[f"logit_{c}"] = lg[:, i].numpy()
    del model, opt, best_state
    gc.collect()
    if device.type == "mps":
        torch.mps.empty_cache()
    elif device.type == "cuda":
        torch.cuda.empty_cache()
    return m_, preds

## 4. Train every run
Runs already in `results.csv` are skipped, so this cell can be re-run after an interruption.

In [ ]:
res_path = os.path.join(OUT, "results.csv")
done = pd.read_csv(res_path) if os.path.exists(res_path) else pd.DataFrame(columns=["run_id"])
t_all = time.time()
for i, (name, protocol, fold, seed) in enumerate(RUNS, 1):
    run_id = f"{name}_{protocol}_f{fold}_s{seed}"
    if run_id in set(done.run_id):
        continue
    t0 = time.time()
    m, preds = run_one(name, protocol, fold, seed, run_id)
    preds.to_csv(os.path.join(OUT, "preds", run_id + ".csv"), index=False)
    m.update(run_id=run_id, model=name, protocol=protocol, fold=fold, seed=seed, minutes=round((time.time() - t0) / 60, 2))
    done = pd.concat([done, pd.DataFrame([m])], ignore_index=True)
    done.to_csv(res_path, index=False)
    print(f"[{i}/{len(RUNS)}] {time.strftime('%H:%M')} {run_id}: test acc {m['accuracy']:.4f} | "
          f"missed tumours, other sources {m.get('missed_tumours_other', np.nan):.3f} | {m['minutes']} min", flush=True)
print(f"training finished in {(time.time() - t_all) / 3600:.1f} h")

## 5. Results
### Mean ± SD over runs (each run's own test fold)

In [ ]:
R = pd.read_csv(res_path)
R = R[R.protocol.isin(PROTOCOLS)]
ALL_MODELS = list(dict.fromkeys(R.model))  # every model in results.csv, so a run of extra models keeps the others
metrics = ["accuracy", "balanced_accuracy", "macro_f1", "auroc_macro", "tumour_sensitivity", "notumor_specificity",
           "missed_tumours_figshare", "missed_tumours_other", "ece_tpal"]
rows = []
for (name, protocol), g in R.groupby(["model", "protocol"], sort=False):
    row = {"model": name, "protocol": protocol, "runs": len(g), "params (M)": round(g.params_m.iloc[0], 2),
           "min/run": round(g.minutes.mean(), 1)}
    row.update({c: f"{g[c].mean():.4f} ± {g[c].std(ddof=1) if len(g) > 1 else 0:.4f}" for c in metrics})
    rows.append(row)
if TPAL_DIR and os.path.exists(os.path.join(TPAL_DIR, "results.csv")):
    Tr = pd.read_csv(os.path.join(TPAL_DIR, "results.csv"))
    Tr = Tr[Tr.run_id.str.startswith("grouped_5fold") & Tr.seed.isin(SEEDS) & Tr.fold.isin(FOLDS)]
    if len(Tr):
        row = {"model": "T-PAL (re-run)", "protocol": "grouped_5fold", "runs": len(Tr), "params (M)": 0.61,
               "min/run": round(Tr.minutes.mean(), 1)}
        row.update({c: f"{Tr[c].mean():.4f} ± {Tr[c].std(ddof=1) if len(Tr) > 1 else 0:.4f}" for c in metrics if c in Tr})
        rows.append(row)
table_models = pd.DataFrame(rows).set_index(["model", "protocol"])
table_models.to_csv(os.path.join(OUT, "table_models.csv"))
table_models

### Paired comparison with T-PAL (leakage-free folds)
For each seed with every selected fold done (all five: 7,200 images), the model's out-of-fold
predictions are matched image by image with T-PAL's. Δ = model − T-PAL accuracy, with a 95% CI and p-value from a
paired bootstrap over families, plus an exact McNemar test.

In [ ]:
acc = lambda y, p: float((p.argmax(1) == y).mean())
pcols = [f"p_{c}" for c in classes]

def pooled(fmt):  # out-of-fold predictions for the selected folds, or None if any fold is missing
    fs = [fmt.format(k) for k in FOLDS]
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if all(map(os.path.exists, fs)) else None

rows = []
if TPAL_DIR:
    for name in ALL_MODELS:
        for seed in SEEDS:
            a = pooled(os.path.join(OUT, "preds", f"{name}_grouped_5fold_f{{}}_s{seed}.csv"))
            b = pooled(os.path.join(TPAL_DIR, "preds", f"grouped_5fold_f{{}}_s{seed}.csv"))
            if a is None or b is None:
                continue
            j = a.merge(b[["rel_path"] + pcols], on="rel_path", suffixes=("_m", "_t"))
            y, fam = j.y.values, j.family.values
            Pm, Pt = j[[c + "_m" for c in pcols]].values, j[[c + "_t" for c in pcols]].values
            d, lo, hi, p = ev.paired_bootstrap_diff(acc, y, Pm, Pt, groups=fam, n_boot=1000)
            n_mt, n_tm, p_mc = ev.mcnemar_exact(y, Pm.argmax(1), Pt.argmax(1))
            rows.append({"model": name, "seed": seed, "images": len(j), "model acc": round(acc(y, Pm), 4),
                         "T-PAL acc": round(acc(y, Pt), 4), "Δ [95% CI]": f"{d:+.4f} [{lo:+.4f}, {hi:+.4f}]",
                         "bootstrap p": round(p, 4), "only model right": n_mt, "only T-PAL right": n_tm,
                         "McNemar p": float(f"{p_mc:.3g}")})
paired = pd.DataFrame(rows).set_index(["model", "seed"]) if rows else pd.DataFrame()
paired.to_csv(os.path.join(OUT, "table_vs_tpal.csv"))
paired

### Missed tumours by image source
Every no-tumour image comes from a non-figshare source. A model that learned the source rather than
the lesion misses tumours on non-figshare images far more often. The 76 non-figshare gliomas (all
in the *Testing* folder) are the hardest case for T-PAL.

In [ ]:
rows = []
for name in ALL_MODELS:
    fs = sorted(glob.glob(os.path.join(OUT, "preds", f"{name}_grouped_5fold_f*_s*.csv")))
    if not fs:
        continue
    P = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
    pred = np.array(classes)[P[pcols].values.argmax(1)]
    lab = np.array(classes)[P.y.values]
    tum = lab != "notumor"
    other_gl = (lab == "glioma") & (P.source.values == "other")
    rows.append({"model": name, "predictions": len(P),
                 "missed tumours, figshare": round(float((pred[tum & (P.source.values == "figshare")] == "notumor").mean()), 4),
                 "missed tumours, other sources": round(float((pred[tum & (P.source.values == "other")] == "notumor").mean()), 4),
                 "non-figshare gliomas correct": round(float((pred[other_gl] == "glioma").mean()), 3),
                 "non-figshare gliomas called no tumour": round(float((pred[other_gl] == "notumor").mean()), 3)})
by_source = pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()
by_source.to_csv(os.path.join(OUT, "table_by_source.csv"))
by_source

In [ ]:
import matplotlib.pyplot as plt

G = R[R.protocol == "grouped_5fold"]
names = list(dict.fromkeys(G.model))
fig, ax = plt.subplots(figsize=(max(7, 0.7 * len(names) + 2), 4.5))
for i, n in enumerate(names):
    v = G.loc[G.model == n, "accuracy"].values
    ax.scatter(np.full(len(v), i) + np.random.default_rng(i).uniform(-0.12, 0.12, len(v)), v, s=18, alpha=0.75)
    ax.hlines(v.mean(), i - 0.3, i + 0.3, color="black")
if TPAL_DIR and os.path.exists(os.path.join(TPAL_DIR, "results.csv")):
    Tr = pd.read_csv(os.path.join(TPAL_DIR, "results.csv"))
    t = Tr.loc[Tr.run_id.str.startswith("grouped_5fold"), "accuracy"].mean()
    ax.axhline(t, ls="--", color="grey", label=f"T-PAL, leakage-free folds ({t:.1%})")
    ax.legend(loc="lower left")
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=40, ha="right")
ax.set_ylabel("Test accuracy per run")
ax.set_title("Leakage-free 5-fold: BiPAL and baselines")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "accuracy_by_model.png"), dpi=200)
plt.show()

In [ ]:
if os.path.isdir("/kaggle/working"):  # on Kaggle, zip the output for download
    import shutil
    print("Saved", shutil.make_archive(OUT, "zip", OUT))
print(sorted(os.listdir(OUT)))